---
## 18. 🕹️ Arcade de datos: juegos clásicos y bases de datos

El **Arcade de datos** tiene tres juegos clásicos. Cada uno enseña una idea distinta y todos guardan sus partidas en la misma base de datos:

| Juego | Idea de programación | Inteligencia artificial o visión |
|---|---|---|
| Culebrita | bucle `while`, listas (agregar cabeza, quitar cola) | búsqueda en anchura (BFS) que encuentra el camino a la galleta |
| Rompeladrillos | bucle `for` dentro de otro bucle, condicionales de choque | piloto que juega **mirando los píxeles** con un rango de color HSV, como OpenCV |
| Triqui | revisar 8 líneas con un bucle y un condicional | tres algoritmos: al azar, reglas `if/elif/else` y minimax |

### El modelo de datos del arcade
Son tres tablas. `partidas` tiene dos claves foráneas: cada partida pertenece a **un** jugador y a **un** juego, y cada jugador o juego tiene **muchas** partidas (relaciones 1 a N).

```
jugadores (id PK, nombre, curso)
juegos    (id PK, nombre, categoria)
partidas  (id PK, jugador_id FK → jugadores, juego_id FK → juegos,
           puntaje, duracion_s, resultado, piloto, fecha)
```
`resultado` vale `'gano'`, `'perdio'` o `'empate'`. `piloto` vale `'humano'` o `'ia'` (cuando jugó el piloto automático).

### En el arcade también hay MVC
| Parte | En el arcade |
|---|---|
| Modelo | el objeto `Modelo` con el estado de cada juego y la base de datos |
| Vista | las funciones que dibujan la pantalla y los paneles |
| Controlador | los manejadores del teclado, el ratón y los botones; al terminar una partida arman el `INSERT` |

La celda siguiente crea las tres tablas con datos de ejemplo de 8 jugadores (inventados) y 105 partidas, los mismos que trae el arcade.

In [1]:
random.seed(91)
jugadores_arcade = [
    (1, "Valentina", 9), (2, "Santiago", 10), (3, "Mariana", 11), (4, "Samuel", 9),
    (5, "Isabella", 10), (6, "Matías", 11), (7, "Sofía", 10), (8, "Juan José", 9),
]
juegos_arcade = [(1, "Culebrita", "arcade"), (2, "Rompeladrillos", "arcade"), (3, "Triqui", "tablero")]

partidas_arcade = []
habilidad = {j[0]: random.uniform(0.6, 1.4) for j in jugadores_arcade}
for jugador_id, nombre, curso in jugadores_arcade[:-1]:          # Juan José todavía no ha jugado
    for juego_id in (1, 2, 3):
        intentos = random.randint(2, 6)
        primer_dia = random.randint(1, 6)
        for intento in range(1, intentos + 1):
            practica = 1 + 0.22 * (intento - 1)                    # cada intento juega un poco mejor
            dia = min(28, primer_dia + 4 * (intento - 1) + random.randint(0, 2))   # cada intento es unos días después
            if juego_id == 1:
                galletas = max(1, int(random.gauss(7, 1.4) * habilidad[jugador_id] * practica))
                galletas = min(galletas, 25)
                puntaje, duracion = galletas * 10, int(galletas * random.uniform(4, 7) + 8)
                resultado = "gano" if galletas == 25 else "perdio"
            elif juego_id == 2:
                ladrillos = min(48, max(2, int(random.gauss(15, 3) * habilidad[jugador_id] * practica)))
                puntaje, duracion = ladrillos * 10, int(ladrillos * random.uniform(2, 3.5) + 20)
                resultado = "gano" if ladrillos == 48 else "perdio"
                if resultado == "gano":
                    puntaje += 100
            else:
                r = random.random() * habilidad[jugador_id] * (0.85 + 0.1 * intento)
                resultado = "gano" if r > 0.85 else "empate" if r > 0.45 else "perdio"
                puntaje = {"gano": 100, "empate": 50, "perdio": 0}[resultado]
                duracion = random.randint(15, 60)
            partidas_arcade.append((jugador_id, juego_id, puntaje, duracion, resultado, "humano", f"2026-10-{dia:02d}"))
# El piloto automático también juega: BFS en la culebrita y visión en el rompeladrillos
for _ in range(6):
    galletas = random.randint(18, 25)
    partidas_arcade.append((3, 1, galletas * 10, galletas * 4 + 6, "gano" if galletas == 25 else "perdio", "ia", f"2026-10-{random.randint(1, 28):02d}"))
for _ in range(5):
    ladrillos = random.randint(30, 48)
    partidas_arcade.append((6, 2, ladrillos * 10 + (100 if ladrillos == 48 else 0), ladrillos * 2 + 15, "gano" if ladrillos == 48 else "perdio", "ia", f"2026-10-{random.randint(1, 28):02d}"))
partidas_arcade.sort(key=lambda p: p[6])                           # en orden de fecha

DDL_ARCADE = """
CREATE TABLE jugadores (
  id INTEGER PRIMARY KEY {AUTO},
  nombre VARCHAR(40) NOT NULL,
  curso INTEGER
);
CREATE TABLE juegos (
  id INTEGER PRIMARY KEY {AUTO},
  nombre VARCHAR(30) NOT NULL,
  categoria VARCHAR(20)
);
CREATE TABLE partidas (
  id INTEGER PRIMARY KEY {AUTO},
  jugador_id INTEGER NOT NULL,
  juego_id INTEGER NOT NULL,
  puntaje INTEGER,
  duracion_s INTEGER,
  resultado VARCHAR(10),
  piloto VARCHAR(10),
  fecha DATE,
  FOREIGN KEY (jugador_id) REFERENCES jugadores(id),
  FOREIGN KEY (juego_id) REFERENCES juegos(id)
)
"""
COLUMNAS_ARCADE = {
    "jugadores": ["id", "nombre", "curso"],
    "juegos": ["id", "nombre", "categoria"],
    "partidas": ["jugador_id", "juego_id", "puntaje", "duracion_s", "resultado", "piloto", "fecha"],
}
DATOS_ARCADE = {"jugadores": jugadores_arcade, "juegos": juegos_arcade, "partidas": partidas_arcade}

def crear_tablas_arcade():
    for tabla in ["partidas", "juegos", "jugadores"]:
        sql(f"DROP TABLE IF EXISTS {tabla}", mensaje=False)
    for sentencia in DDL_ARCADE.format(AUTO="" if MOTOR == "sqlite" else "AUTO_INCREMENT").split(";"):
        if sentencia.strip():
            sql(sentencia, mensaje=False)
    for tabla, columnas in COLUMNAS_ARCADE.items():
        if MOTOR == "sqlite":
            marcas = ", ".join("?" * len(columnas))
            conexion.executemany(f"INSERT INTO {tabla} ({', '.join(columnas)}) VALUES ({marcas})", DATOS_ARCADE[tabla])
            conexion.commit()
        else:
            from sqlalchemy import text
            marcas = ", ".join(":" + c for c in columnas)
            with motor_mysql.begin() as c:
                c.execute(text(f"INSERT INTO {tabla} ({', '.join(columnas)}) VALUES ({marcas})"),
                          [dict(zip(columnas, fila)) for fila in DATOS_ARCADE[tabla]])
    print("Tablas del arcade listas:", {t: int(sql(f"SELECT COUNT(*) AS n FROM {t}")["n"][0]) for t in COLUMNAS_ARCADE})

crear_tablas_arcade()

NameError: name 'random' is not defined

Mira las tablas y una primera consulta con `JOIN`: el nombre del jugador y del juego no están en `partidas`; se traen desde sus tablas usando las claves foráneas.

In [ ]:
display(sql("SELECT * FROM juegos"))
display(sql("SELECT * FROM jugadores"))
sql("""
SELECT p.fecha, j.nombre AS jugador, g.nombre AS juego, p.puntaje, p.resultado, p.piloto
FROM partidas p
JOIN jugadores j ON p.jugador_id = j.id
JOIN juegos g    ON p.juego_id = g.id
ORDER BY p.fecha DESC
LIMIT 8
""")

### 🧩 Reto R13: Triqui ganados
Muestra `id`, `jugador_id` y `fecha` de las partidas de **Triqui** (`juego_id = 3`) que el jugador **ganó**.

In [ ]:
mi_consulta = """
SELECT id, jugador_id, fecha FROM partidas WHERE juego_id = 3 AND resultado = 'gano'
"""
revisar("R13", mi_consulta)

### 🧩 Reto R14: el salón de la fama
Muestra el **nombre del jugador**, el **nombre del juego** y el **puntaje** de todas las partidas con 350 puntos o más, de mayor a menor puntaje. Necesitas unir las tres tablas.

In [ ]:
mi_consulta = """
SELECT j.nombre AS jugador, g.nombre AS juego, p.puntaje
FROM partidas p
JOIN jugadores j ON p.jugador_id = j.id
JOIN juegos g ON p.juego_id = g.id
WHERE p.puntaje >= 350
ORDER BY p.puntaje DESC
"""
revisar("R14", mi_consulta)

### 🧩 Reto R15: ¿quién juega mejor, la persona o la IA?
Para la Culebrita y el Rompeladrillos (`juego_id IN (1, 2)`), muestra el **nombre del juego**, el **piloto** y el **promedio de puntaje** redondeado a 1 decimal. Agrupa por juego y por piloto.

In [ ]:
mi_consulta = """
SELECT g.nombre, p.piloto, ROUND(AVG(p.puntaje), 1) AS promedio
FROM partidas p
JOIN juegos g ON p.juego_id = g.id
WHERE p.juego_id IN (1, 2)
GROUP BY g.nombre, p.piloto
"""
revisar("R15", mi_consulta)

### 🧩 Reto R16: jugadores que nunca han jugado
Muestra el **nombre** de los jugadores que no tienen ninguna partida. (Pista: `LEFT JOIN` + `IS NULL`.)

In [ ]:
mi_consulta = """
SELECT j.nombre
FROM jugadores j
LEFT JOIN partidas p ON p.jugador_id = j.id
WHERE p.id IS NULL
"""
revisar("R16", mi_consulta)

### 🧩 Reto R17: los más constantes
Muestra el **nombre** de los jugadores con **más de 15 partidas** y cuántas tienen (llama a esa columna `partidas`), de mayor a menor. (Pista: `GROUP BY` y `HAVING`.)

In [ ]:
mi_consulta = """
SELECT j.nombre, COUNT(*) AS partidas
FROM partidas p
JOIN jugadores j ON p.jugador_id = j.id
GROUP BY j.id, j.nombre
HAVING COUNT(*) > 15
ORDER BY partidas DESC
"""
revisar("R17", mi_consulta)

### Gráficas: ¿se aprende jugando?
Con pandas numeramos los intentos de cada jugador en cada juego (`cumcount` cuenta las filas dentro de cada grupo) y calculamos el puntaje promedio del primer intento, del segundo, etc. Si la línea sube, los jugadores mejoran con la práctica. Solo graficamos hasta el cuarto intento, porque pocos jugadores tienen más y un promedio de dos o tres partidas engaña.

In [ ]:
df = sql("""
SELECT p.jugador_id, g.nombre AS juego, p.fecha, p.id, p.puntaje, p.piloto
FROM partidas p JOIN juegos g ON p.juego_id = g.id
WHERE p.piloto = 'humano'
ORDER BY p.fecha, p.id
""")
df["intento"] = df.groupby(["jugador_id", "juego"]).cumcount() + 1          # 1, 2, 3… por jugador y juego
curva = df[df["intento"] <= 4].groupby(["juego", "intento"])["puntaje"].mean().unstack(0)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for juego, color in zip(["Culebrita", "Rompeladrillos", "Triqui"], [AGUA, NARANJA, AZUL]):
    ax.plot(curva.index, curva[juego], marker="o", lw=2, color=color, label=juego)
    ax.text(curva.index[-1] + 0.08, curva[juego].iloc[-1], juego, va="center", fontsize=9)
ax.set_xticks(range(1, 5)); ax.set_xlim(0.8, 4.9)
ax.set_xlabel("número de intento"); ax.set_ylabel("puntaje promedio")
ax.set_title("Curva de aprendizaje en el arcade")
plt.show()

In [ ]:
# Humano contra piloto automático (barras agrupadas)
df = sql("""
SELECT g.nombre AS juego, p.piloto, AVG(p.puntaje) AS promedio
FROM partidas p JOIN juegos g ON p.juego_id = g.id
WHERE p.juego_id IN (1, 2)
GROUP BY g.nombre, p.piloto
""")
tabla = df.pivot(index="juego", columns="piloto", values="promedio")
fig, ax = plt.subplots(figsize=(6.5, 3.8))
x = np.arange(len(tabla))
for k, (piloto, color) in enumerate([("humano", AZUL), ("ia", NARANJA)]):
    barras = ax.bar(x + (k - 0.5) * 0.36, tabla[piloto], width=0.34, color=color, label="persona" if piloto == "humano" else "piloto automático")
    etiquetar_barras(ax, barras)
ax.set_xticks(x, tabla.index); ax.set_ylabel("puntaje promedio"); ax.grid(axis="x", visible=False)
ax.legend(frameon=False); ax.set_title("¿Quién juega mejor?")
plt.show()

### Los juegos en Python: bucles y condicionales
Ahora vas a programar el corazón de cada juego. Son las mismas reglas que viste resaltadas en el panel **Algoritmo** del arcade.

#### 🧩 Reto J1: ¿quién ganó el Triqui?
El tablero es una lista de 9 casillas numeradas así:
```
0 | 1 | 2
3 | 4 | 5
6 | 7 | 8
```
Cada casilla vale `"X"`, `"O"` o `""` (vacía). Escribe `ganador_triqui(tablero)` que devuelva `"X"` u `"O"` si alguien tiene tres en línea, `"empate"` si el tablero está lleno sin ganador, o `None` si el juego sigue. Usa un bucle `for` sobre la lista `LINEAS`.

In [ ]:
LINEAS = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]

def ganador_triqui(tablero):
    for a, b, c in LINEAS:
        if tablero[a] != "" and tablero[a] == tablero[b] == tablero[c]:
            return tablero[a]
    if "" not in tablero:
        return "empate"
    return None

comprobar("J1", ganador_triqui)

#### 🧩 Reto J2: el rebote de la pelota
En el Rompeladrillos, en cada vuelta del bucle la pelota avanza `vx` en x y `vy` en y. Escribe `paso_pelota(x, y, vx, vy, ancho)` que devuelva la tupla `(x_nueva, y_nueva, vx, vy)`:
* calcula `x_nueva = x + vx` y `y_nueva = y + vy`;
* si `x_nueva` es menor que 0 o mayor que `ancho`, invierte `vx` (rebota en la pared lateral);
* si `y_nueva` es menor que 0, invierte `vy` (rebota en el techo).

In [ ]:
def paso_pelota(x, y, vx, vy, ancho):
    x_nueva, y_nueva = x + vx, y + vy
    if x_nueva < 0 or x_nueva > ancho:
        vx = -vx
    if y_nueva < 0:
        vy = -vy
    return (x_nueva, y_nueva, vx, vy)

comprobar("J2", paso_pelota)

#### 🧩 Reto J3: un paso de la culebrita (difícil)
El cuerpo es una lista de tuplas `(x, y)` con la cabeza primero; el tablero mide 20 × 20 (coordenadas de 0 a 19). Escribe `avanzar_culebrita(cuerpo, direccion, galleta)`:
1. calcula la nueva cabeza sumando `direccion` (por ejemplo `(1, 0)` es a la derecha);
2. si se sale del tablero o choca con su cuerpo, devuelve `None`;
3. si no, devuelve una **lista nueva** con la nueva cabeza al frente y **sin la cola**, excepto si la nueva cabeza está en la `galleta`: entonces conserva la cola (la culebrita crece).

Pista: revisa el choque contra `cuerpo[:-1]` si no come, porque la cola se mueve en el mismo paso.

In [ ]:
def avanzar_culebrita(cuerpo, direccion, galleta):
    cabeza = (cuerpo[0][0] + direccion[0], cuerpo[0][1] + direccion[1])
    come = cabeza == galleta
    resto = cuerpo if come else cuerpo[:-1]
    if not (0 <= cabeza[0] < 20 and 0 <= cabeza[1] < 20) or cabeza in resto:
        return None
    return [cabeza] + list(resto)

comprobar("J3", avanzar_culebrita)

### Pega aquí tus partidas del arcade
1. En el arcade, escoge tu nombre en **Jugador** (o crea uno con «+ Nuevo jugador») y juega varias partidas de los tres juegos. Prueba también el piloto automático.
2. En la pestaña **Base de datos** toca **Copiar SQL para el cuaderno**.
3. Pega el texto entre las comillas triples y ejecuta la celda.

Tus jugadores nuevos empiezan en el id 101 y tus partidas en el 1001, así que no se mezclan con los datos de ejemplo.

In [ ]:
INSERT INTO jugadores (id, nombre, curso) VALUES (101, 'Invitado', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (102, 'Stev Arismendi', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (103, 'Compañero de Clase', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (104, 'Compañero de Clase', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (105, 'Compañero de Clase', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (106, 'Compañero de Clase', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (107, 'Compañero de Clase', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (108, 'Compañero de Clase', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (109, 'Compañero de Clase', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (110, 'Compañero de Clase', 10);
INSERT INTO jugadores (id, nombre, curso) VALUES (111, 'Compañero de Clase', 10);
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1001, 101, 1, 0, 3, 'perdio', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1002, 101, 1, 0, 1, 'perdio', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1003, 101, 1, 0, 2, 'perdio', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1004, 101, 1, 0, 1, 'perdio', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1005, 102, 1, 250, 63, 'gano', 'ia', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1006, 102, 3, 100, 544, 'gano', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1007, 102, 3, 100, 5, 'gano', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1008, 102, 3, 0, 3, 'perdio', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1009, 102, 3, 100, 3, 'gano', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1010, 102, 3, 100, 3, 'gano', 'humano', '2026-11-06');
INSERT INTO partidas (id, jugador_id, juego_id, puntaje, duracion_s, resultado, piloto, fecha) VALUES (1011, 102, 3, 100, 6, 'gano', 'humano', '2026-11-06');

**Para investigar con tus datos:** ¿en qué juego mejoraste más entre tu primera y tu última partida? ¿Tu mejor puntaje en la Culebrita le gana al promedio del piloto automático? Escribe la consulta y haz una gráfica con tus resultados.

Si quieres volver a los datos de ejemplo del arcade, ejecuta `crear_tablas_arcade()`.

In [ ]:
# 1. Mejora por juego (primera vs última partida)
df_mejora = sql("""
SELECT g.nombre AS juego,
       FIRST_VALUE(p.puntaje) OVER (PARTITION BY p.juego_id ORDER BY p.fecha) AS primera,
       LAST_VALUE(p.puntaje) OVER (PARTITION BY p.juego_id ORDER BY p.fecha
           ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS ultima
FROM partidas p
JOIN juegos g ON p.juego_id = g.id
WHERE p.jugador_id = 101 AND p.piloto = 'humano'
""").drop_duplicates()
df_mejora["mejora"] = df_mejora["ultima"] - df_mejora["primera"]
display(df_mejora)

# 2. Comparación Culebrita: tu mejor vs promedio del piloto
df_comp = sql("""
SELECT
    (SELECT MAX(puntaje) FROM partidas WHERE jugador_id = 101 AND juego_id = 1 AND piloto = 'humano') AS mi_mejor,
    (SELECT ROUND(AVG(puntaje), 1) FROM partidas WHERE juego_id = 1 AND piloto = 'ia') AS promedio_ia
""")
display(df_comp)

# Gráfica comparativa
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Mi mejor puntaje (humano)", "Promedio piloto IA"],
       [df_comp["mi_mejor"][0], df_comp["promedio_ia"][0]],
       color=[AZUL, NARANJA])
ax.set_ylabel("Puntaje")
ax.set_title("Culebrita: humano vs piloto automático")
plt.show()